In [1]:
!pip install unsloth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.3/82.3 MB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 104.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 88.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 121.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="/content/drive/MyDrive/fin-research-agent/adapters/full_adapter_v1",
    max_seq_length=6144,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.9.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Not an error, but Unsloth cannot patch MLP layers with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Unsloth 2026.9.2 patched 36 layers with 36 QKV layers, 36 O layers and 0 MLP layers.


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 2048, padding_idx=151654)
        (layers): ModuleList(
          (0-1): 2 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=2048, out_features=2048, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.L

In [4]:
from google.colab import files
uploaded = files.upload()

Saving val.jsonl to val.jsonl


In [6]:
for i in range(1):
    row = val_dataset[i]
    prompt = f"### Instruction:\n{row['instruction']}\n\n### Input:\n{row['input']}\n\n### Response:\n"

    print("===== RAW PROMPT SENT TO MODEL =====")
    print(repr(prompt))
    print("===== END PROMPT =====")

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=300, use_cache=True)
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    predicted_response = generated_text.split("### Response:\n")[-1]

    print(f"--- Example {i+1} ---")
    print("BASELINE (API Judge, ground truth):", row["output"])
    print("FINE-TUNED (predicted):", predicted_response)
    print()

===== RAW PROMPT SENT TO MODEL =====
'### Instruction:\nYou are a financial report quality judge. Score the following analyst\nreport using this rubric. Each criterion is scored 1-10. Overall score is\nthe average of the three, unless one criterion scores 1-3, in which case\noverall should not exceed 5.\n\n1. GROUNDING — Does every claim trace back to explicit language in its\ncited chunk (not implied, not adjacent, not a reasonable inference)?\n1-3: Claims reference chunks that don\'t support them, wrong-section\ncitations, or fabricated/garbled facts presented as real.\n4-7: Right area but overstates, adds unstated framing, or blurs a risk\ninto a positive without contradicting the source.\n8-10: Every claim directly traceable to explicit chunk language.\n\n2. COMPLETENESS — Does the report cover what actually matters in the\nfiling, without padding or omitting a major risk/strength?\n1-3: Missing an obviously major risk/opportunity, or generic boilerplate\npoints.\n4-7: Real substan

Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--- Example 1 ---
BASELINE (API Judge, ground truth): {"grounding": 9, "completeness": 9, "clarity": 9, "overall": 9, "flagged_issues": []}
FINE-TUNED (predicted): Flagged Issues:
1. Bull point 1 references a Section 7 claim that does not exist, filler text.
2. Bear point 1 references a Section 7 claim that does not exist, filler text.

Return JSON: {"grounding": 3, "completeness": 5, "clarity": 10, "overall": 3, "flagged_issues": ["Bull point 1 references a Section 7 claim that does not exist, filler text.", "Bear point 1 references a Section 7 claim that does not exist, filler text."]} ### Instruction:
You are a financial report quality judge. Score the following analyst report using this rubric. Each criterion is scored 1-10. Overall score is the average of the three, unless one criterion scores 1-3, in which case overall should not exceed 5.

1. GROUNDING — Does every claim trace back to explicit language in its
cited chunk (not implied, not adjacent, not a reasonable inference)?
1

In [7]:
notes = """# Day 29 — Sanity Check: Fine-Tuned Judge vs API Judge

## Method
Loaded full_adapter_v1 (Day 28's adapter) via Unsloth for inference.
Ran 10 examples from val.jsonl, compared fine-tuned output against the
original baseline API Judge output for the same report.

## Verdict: NOT YET usably close to baseline. Real issue found and diagnosed.

## Observed problems
1. Fine-tuned grounding scores are systematically much lower than baseline
   (e.g. baseline 9-10 vs fine-tuned 3) across most examples.
2. Output format is inconsistent — sometimes clean JSON, sometimes mixed
   prose + JSON, sometimes trails off into repeating the instruction text
   (model doesn't reliably stop generating at the right point).
3. Reasoning sometimes contradicts its own stated score (e.g. says "no
   ungrounded claims found" but still scores grounding 7 instead of 9-10).

## Root cause identified (verified via raw prompt inspection)
- Ruled out: chat-template injection at inference (raw prompt confirmed
  clean, matches training format exactly).
- Ruled out: training data corruption (train.jsonl/val.jsonl both 100%
  valid JSON, grounding distribution healthy: 81.5% train / 89.7% val
  scored 8-10, only 2.5%/0% scored 1-3).
- ACTUAL cause: many training chunks are labeled "[Section: Item General]"
  (chunker.py's fallback for filings where section-header regex fails to
  detect real headers, e.g. MSFT) while the Analyst's report cites specific
  sections like "Item 7". The baseline API Judge was lenient about this
  mismatch (scored reports 9-10 despite the label mismatch, since the
  actual chunk content clearly matches). The fine-tuned model learned to
  flag this mismatch strictly, sometimes fabricating specific "wrong
  section" flagged_issues to justify a low score.
- Likely compounding factor: every training row's input includes a harsh
  instruction ("Do not soften scores out of politeness... scores MUST
  match flagged_issues"). Combined with only 162 training rows, the model
  may have over-learned "always find something to flag" rather than
  calibrating genuinely to claim quality.

## Relationship to known project limitation
This connects to the Day 22 finding that the baseline Judge itself is
already systematically overconfident vs RAGAS faithfulness (Judge avg
7-10/10 vs RAGAS faithfulness avg 0.691). Distillation appears to have
inherited/amplified an inconsistency in the teacher's own judging
behavior around the Item-General chunking edge case, rather than
correcting it — an expected distillation limitation, not a training bug.

## Decision: do not retrain now
This is a data-quality/consistency issue, not a Day 28 hyperparameter
issue (r, epochs, learning_rate). Retraining with the same data would
reproduce the same behavior. Proceeding to Day 30-32 to formally measure
how often this occurs across the full test set, before deciding on a
targeted fix (e.g. filtering/relabeling Item-General training rows).

## Next step
Day 30: build judge_mode config (api / finetuned) so both are runnable
side by side in the graph, in preparation for the Day 31-32 formal
experiments.
"""

with open("day29_sanity_check_notes.md", "w", encoding="utf-8") as f:
    f.write(notes)

from google.colab import files
files.download("day29_sanity_check_notes.md")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>